> **[CAPSTONE INTEGRATION TARGET]** Plato's native Main process owns operating-system capabilities; its isolated Renderer displays physical-shadow telemetry through a small preload API. This lab identifies who owns each process before automating it.

# Phase 1 — Main, Renderer, and an honest automation boundary

Tasks: VF-002, VF-004, VF-100, VF-101. Run every cell top to bottom. The default fixture never opens Electron or the network. A fixture result is evidence about the teaching model, not proof of Windows behavior. Live execution is an explicit final exercise.


## Predict before running

Write your answers: Which process may create a native BrowserWindow? Which process owns the DOM? Does a CDP port grant the renderer `require('fs')`? What must an MCP client inspect before calling a tool?

Topology: Linux notebook → stdio MCP process → SSH-tunneled renderer CDP on Windows, or Linux notebook → reviewed SSH stdio command → Windows electron-driver → **owned** Electron process. Only the second route provides the driver's Main evaluation capability. An HTTP `/json/version` discovery endpoint is CDP, not an invented MCP `/mcp` endpoint.


In [ ]:
from pathlib import Path
import sys, os, json, asyncio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts" / "notebook_support.py").exists()), None)
assert ROOT is not None, "Open this notebook inside the supplied VoltForge source bundle"
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import notebook_support as lab
LIVE = os.environ.get("VOLTFORGE_LIVE_MCP") == "1"
print(json.dumps({"event": "lab.mode", "live_opt_in": LIVE, "workspace": ROOT.name}))


## Discover capabilities instead of assuming names

`fixture_*` are synthetic names owned by this repository. Upstream adapters have different names and schemas. Inspect both `name` and `inputSchema`; the broker refuses tools that have not been discovered.


In [ ]:
broker = lab.MCPBroker(lab.OfflineMCP(), timeout_seconds=1)
catalog = await broker.discover()
for tool in catalog:
    print(json.dumps({"name": tool["name"], "inputSchema": tool["inputSchema"]}, indent=2))
assert len(catalog) == 3
try:
    await broker.call("guessed_upstream_tool", {})
    raise AssertionError("Undiscovered tool unexpectedly accepted")
except lab.ValidationError as error:
    print("Expected rejection:", error)


## Follow ownership through the fixture

The actual build entry is `dist/main.js`. The fixture returns the secure Main policy, then mutates a text label in its model Renderer. Tool evaluation belongs to the privileged test harness. It is never added to the public `window.plato` bridge.


In [ ]:
spawn = await broker.call("fixture_spawn", {"entry": "dist/main.js"})
policy = await broker.call("fixture_main", {})
render = await broker.call("fixture_renderer", {"text": "Plato | shadow pipeline ready"})
assert spawn["process"] == "main" and spawn["windowId"] == 1
assert policy["contextIsolation"] is True
assert policy["nodeIntegration"] is False and policy["sandbox"] is True
assert render["textContent"] == "Plato | shadow pipeline ready"
assert render["nodeAvailable"] is False
print(json.dumps({"mode": "offline-fixture", "spawn": spawn, "policy": policy, "renderer": render}, indent=2))


## Text is data; transport can fail

The next two tests falsify accidental HTML execution in the fixture and an indefinitely awaited tool. Real DOM security must also be inspected in Electron; a dictionary cannot enforce Chromium's sandbox.


In [ ]:
text = "<img src=x onerror=alert('inert-lab-string')>"
render = await broker.call("fixture_renderer", {"text": text})
assert render["textContent"] == text and render["htmlExecuted"] is False
class DelayedFixture(lab.OfflineMCP):
    async def list_tools(self):
        await asyncio.sleep(0.1)
logs = []
slow = lab.MCPBroker(DelayedFixture(), timeout_seconds=0.01, logger=logs.append)
try:
    await slow.discover()
    raise AssertionError("Expected discovery deadline")
except lab.LabTimeout:
    pass
assert json.loads(logs[-1])["event"] == "mcp.discover.timeout"
print("\n".join(logs))


## Optional live rehearsal (review first)

Read the phase README. First run `notebook_support.py discover --live --config YOUR_CONFIG` to inspect actual schemas. Then add reviewed role mappings. `electron-driver` requires `spawn`, `main`, `renderer`, and `stop`; `laststance` accepts only `renderer` and attaches to an existing application. Start/stop an owned app in the Windows interactive session. Use a harmless Main expression such as `({ version: app.getVersion(), windows: BrowserWindow.getAllWindows().length })`, adapted to the **discovered tool's evaluation context**. For the Renderer, inspect `typeof require`, `typeof process`, and mutate an existing text label with `textContent`.

Set `VOLTFORGE_LIVE_MCP=1` and `VOLTFORGE_MCP_CONFIG` before launching the kernel. Never place keys or private endpoint details in saved notebook outputs. A client deadline is cooperative; SDK cancellation/transport cleanup may extend shutdown. Inspect and terminate the owned Windows process if it wedges. This exercise never guarantees a remote hard kill.


In [ ]:
if LIVE:
    live_result = await lab.architecture_lab()
    assert live_result["mode"] == "live"
    print(json.dumps(live_result, indent=2))
else:
    print(json.dumps({"event": "live.skipped", "reason": "explicit opt-in absent", "fixture_passed": True}))


## Evidence and transfer

Record the discovered tool names, exact argument schemas, adapter/version, host owning each process, native window observation, harmless Main value, Renderer text before/after, and cleanup result. Mark the live rows `not run` until observed. Explain why a laststance renderer-only trace cannot prove the Main exercise.

Open `../docs/phase-1/index.html`, attempt direct filesystem access, then an allowed mediated request. Compare the simulation with the native process trace. Continue with Phase 2.
